# HealthBot: AI-Powered Patient Education System
### MediTech Solutions — Healthcare AI Capstone

A **LangGraph** workflow that teaches patients about a health topic and then checks their
understanding:

1. Ask the patient for a health topic.
2. The **Groq-hosted model calls the LangChain-community Tavily tool** to search reputable medical sites.
3. Summarize the search results (and *only* the search results) into patient-friendly language.
4. Show the summary and let the patient read it.
5. Wait for the patient to say they are ready for a comprehension check.
6. Generate **one** quiz question answerable from the summary alone.
7. Show the question and collect the patient's answer.
8. Grade the answer (letter grade + justification **with citations from the summary**).
9. Show the grade and feedback.
10. Offer a new topic or exit — **state is fully reset** between topics for privacy.

---

## 1. Environment & API keys

Create a file named `config.env` next to this notebook containing:

```env
GROQ_API_KEY="gsk_**********"
TAVILY_API_KEY="tvly-********"
```

Free keys: Groq at <https://console.groq.com/keys>, Tavily at <https://app.tavily.com/home>.

> **Provider note.** The reference brief uses the OpenAI API. This build runs OpenAI's open-weight
> model `openai/gpt-oss-120b` on **Groq** inference instead. Everything else is unchanged: the model
> still calls the LangChain community Tavily tool through `bind_tools`, and the graph, state and
> prompts are provider-independent. To move to the OpenAI API, replace the two `ChatGroq` lines in
> section 2 with `from langchain_openai import ChatOpenAI` and
> `llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)`, and put `OPENAI_API_KEY` in `config.env`.

In [ ]:
# Load in the Groq key and Tavily key from config.env
from dotenv import load_dotenv
import os

load_dotenv("config.env", override=True)

assert os.getenv("GROQ_API_KEY") is not None, "GROQ_API_KEY is missing from config.env"
assert os.getenv("TAVILY_API_KEY") is not None, "TAVILY_API_KEY is missing from config.env"

print("GROQ_API_KEY loaded:  ", os.getenv("GROQ_API_KEY")[:7] + "...")
print("TAVILY_API_KEY loaded:", os.getenv("TAVILY_API_KEY")[:8] + "...")

In [ ]:
import json
import textwrap
from typing import Annotated, List

from typing_extensions import TypedDict

from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    RemoveMessage,
    SystemMessage,
    ToolMessage,
)

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

print("Imports OK")

## 2. Model and tools

We use the LangChain **community** Tavily tool (`TavilySearchResults`) and restrict the search to
reputable medical publishers, then **bind the tool to the chat model** so the model itself decides to
issue the search call.

> `langchain-community` emits a deprecation warning for `TavilySearchResults` (the newer home is
> the `langchain-tavily` package). The project brief asks specifically for the **LangChain community
> tool for Tavily**, so that is what is used here; the warning is expected and harmless.

In [ ]:
# Reputable medical sources only - Tavily will not return anything outside this list.
MEDICAL_DOMAINS = [
    "mayoclinic.org",
    "medlineplus.gov",
    "nih.gov",
    "cdc.gov",
    "who.int",
    "clevelandclinic.org",
    "hopkinsmedicine.org",
    "nhs.uk",
    "healthline.com",
]

# LangChain community tool for Tavily search.
tavily_tool = TavilySearchResults(
    max_results=6,
    search_depth="advanced",
    include_answer=False,
    include_raw_content=False,
    include_domains=MEDICAL_DOMAINS,
)

# The Groq-hosted chat model, with the Tavily tool bound to it so the MODEL can call the tool.
CHAT_MODEL = "openai/gpt-oss-120b"   # OpenAI's open-weight model, served by Groq

llm = ChatGroq(model=CHAT_MODEL, temperature=0)
llm_with_tools = llm.bind_tools([tavily_tool])

print("Tool name bound to the model:", tavily_tool.name)

### 2a. Smoke test — prove both APIs work and that the model calls Tavily

The cell below is a sanity check for the rubric ("successful API calls are made to both Tavily and
the chat-model service" / "the model successfully calls Tavily for search"). It is not part of the
graph.

In [ ]:
# 1) Direct Tavily API call
_probe = tavily_tool.invoke({"query": "type 2 diabetes overview"})
print("Tavily returned", len(_probe), "results. First source:", _probe[0]["url"])

# 2) Groq API call in which the MODEL decides to call the Tavily tool
_ai_msg = llm_with_tools.invoke(
    [HumanMessage(content="Search reputable medical sources for information about type 2 diabetes.")]
)
print("Model tool_calls:", json.dumps(_ai_msg.tool_calls, indent=2))

## 3. LangGraph state

`HealthBotState` carries everything the nodes create, read and update. `messages` uses the
`add_messages` reducer so the model always has access to the full conversation — the patient's
request, the model's tool call, the raw Tavily `ToolMessage`, the summary, the quiz question and the
patient's answer.

In [ ]:
class HealthBotState(TypedDict):
    """Shared state passed between every node in the HealthBot graph."""

    # Full conversation, including tool calls and tool results.
    messages: Annotated[list, add_messages]

    topic: str            # health topic the patient asked about
    search_results: str   # flattened Tavily results (text + source URLs)
    sources: List[str]    # source URLs used for the summary
    summary: str          # 3-4 paragraph patient-friendly summary
    quiz_question: str    # single comprehension-check question
    patient_answer: str   # what the patient typed
    grade_feedback: str   # letter grade + justification with citations
    session_active: bool  # True -> loop for a new topic, False -> exit


def blank_state() -> dict:
    """A completely empty HealthBotState payload (used to start and to reset)."""
    return {
        "messages": [],
        "topic": "",
        "search_results": "",
        "sources": [],
        "summary": "",
        "quiz_question": "",
        "patient_answer": "",
        "grade_feedback": "",
        "session_active": True,
    }

## 4. Prompts

Each prompt is deliberately restrictive: the summarizer may use **only** the Tavily results, and the
quiz writer and grader may use **only** the summary.

In [ ]:
SEARCH_SYSTEM_PROMPT = """You are HealthBot, a patient-education assistant built by MediTech Solutions.

You have access to a Tavily web-search tool. When the patient names a health topic or medical
condition, you MUST call the search tool exactly once with a clear, focused query that will surface
reputable medical information (symptoms, causes, diagnosis, treatment, self-care and when to seek
help) from sources such as Mayo Clinic, MedlinePlus, NIH, CDC, WHO, Cleveland Clinic, Johns Hopkins
and the NHS.

Never answer the patient from your own knowledge and never guess. Your only job in this step is to
issue the search tool call."""


SUMMARY_SYSTEM_PROMPT = """You are HealthBot, a patient-education assistant.

Write a patient-friendly explanation of the health topic using ONLY the search results provided in
the user message. You must not add any fact, number, drug name, statistic or recommendation that
does not appear in those search results. If the search results do not cover something, simply leave
it out - do not fill gaps with your own knowledge.

Rules:
- Exactly 3 to 4 paragraphs of flowing prose. No bullet points, no headings, no markdown.
- Write at roughly a 6th-to-8th grade reading level. Use short sentences and everyday words.
- The first time you use a medical term, explain it in plain language right after it.
- Suggested flow: (1) what the condition is and who it affects, (2) common signs, causes and how it
  is diagnosed, (3) how it is usually treated or managed day to day, (4) warning signs that mean the
  patient should contact a clinician.
- Be calm and reassuring. Do not diagnose the patient or give them a personal treatment plan.
- End the final paragraph with a short reminder that this is general education and that they should
  talk to their own healthcare provider about their situation."""


QUIZ_SYSTEM_PROMPT = """You are HealthBot's comprehension-check writer.

Using ONLY the patient-education summary provided in the user message, write EXACTLY ONE open-ended
quiz question that checks whether the patient understood a key idea in the summary.

Rules:
- The question must be fully answerable from the summary alone. Do not require any outside knowledge.
- Ask about something meaningful (a cause, a warning sign, a treatment, a self-care step) - not a
  trivia detail or an exact number unless that number is central to the summary.
- One sentence. Plain language. No multiple-choice options, no answer key, no preamble.
- Output only the question text and nothing else."""


GRADER_SYSTEM_PROMPT = """You are HealthBot's comprehension-check grader.

You will be given (1) the patient-education summary, (2) the quiz question, and (3) the patient's
answer. Grade the patient's answer using ONLY the summary as your source of truth. Do not penalise
the patient for anything the summary never said, and do not use outside medical knowledge.

Respond in exactly this format:

Grade: <a single letter grade: A, B, C, D or F>

Justification: <2-4 sentences explaining why that grade was given. State what the patient got right
and what they missed or got wrong.>

Citations from the summary: <one to three short direct quotations from the summary, each wrapped in
quotation marks, that support your grade and reinforce the correct answer.>

Be encouraging and constructive - the goal is to help the patient learn, not to discourage them."""

In [ ]:
def banner(title: str) -> None:
    """Small helper so the patient-facing output is easy to read in the notebook."""
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)


def wrap(text: str, width: int = 78) -> str:
    """Wrap prose paragraphs so long summaries stay readable in a notebook."""
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    return "\n\n".join(textwrap.fill(p, width=width) for p in paragraphs)

## 5. Nodes

Every node has a single responsibility, reads what it needs from state, and returns only the state
keys it changes.

| Node | Responsibility |
| --- | --- |
| `collect_topic` | Ask the patient (`input()`) what they want to learn about |
| `search_agent` | The chat model decides to call the Tavily tool |
| `tavily_tools` | `ToolNode` that actually executes the Tavily search |
| `collect_results` | Flatten the raw Tavily `ToolMessage` into state |
| `summarize` | Turn the search results into a 3-4 paragraph patient-friendly summary |
| `display_summary` | Print the summary and its sources |
| `readiness_check` | Wait for the patient to say they are ready for the quiz |
| `generate_quiz` | Create one question from the summary only |
| `present_quiz` | Print the quiz question |
| `collect_answer` | Collect the patient's answer (`input()`) |
| `grade_answer` | Grade with a letter grade + justification + citations |
| `present_grade` | Print the grade and feedback |
| `ask_next_action` | Ask: new topic or exit? |
| `reset_state` | Wipe all patient/topic data before starting a new topic |
| `end_session` | Say goodbye |

In [ ]:
# ---------- Node 1: collect the health topic from the patient ----------
def collect_topic(state: HealthBotState) -> dict:
    banner("HEALTHBOT - PATIENT EDUCATION")
    topic = ""
    while not topic:
        topic = input("What health topic or medical condition would you like to learn about? ").strip()
        if not topic:
            print("Please enter a health topic so I know what to look up.")

    print(f"\nThanks! Searching reputable medical sources for: {topic}\n")

    return {
        "topic": topic,
        "session_active": True,
        "messages": [
            SystemMessage(content=SEARCH_SYSTEM_PROMPT),
            HumanMessage(content=f"I would like to learn about: {topic}"),
        ],
    }


# ---------- Node 2: the model decides to call Tavily ----------
def search_agent(state: HealthBotState) -> dict:
    """The Groq-hosted model reads the conversation and emits a Tavily tool call."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


# ---------- Node 3: execute the Tavily tool call ----------
tavily_tool_node = ToolNode([tavily_tool])


# ---------- Node 4: pull the Tavily results out of the ToolMessage ----------
def collect_results(state: HealthBotState) -> dict:
    """Flatten every ToolMessage from this turn into plain text + a source list."""
    # Walk backwards only as far as the patient's request for THIS topic, then restore order.
    tool_messages = []
    for msg in reversed(state["messages"]):
        if isinstance(msg, HumanMessage):
            break  # we have walked back to the start of this topic
        if isinstance(msg, ToolMessage):
            tool_messages.append(msg)
    tool_messages.reverse()

    chunks, sources = [], []
    for msg in tool_messages:
        payload = msg.content
        if isinstance(payload, str):
            try:
                payload = json.loads(payload)
            except json.JSONDecodeError:
                payload = [{"url": "Tavily", "title": "", "content": msg.content}]

        for result in payload:
            url = result.get("url", "unknown source")
            sources.append(url)
            chunks.append(
                f"SOURCE: {url}\n"
                f"TITLE: {result.get('title', '')}\n"
                f"CONTENT: {result.get('content', '')}"
            )

    print(f"Found {len(sources)} results from reputable medical sources.")
    print("Writing a patient-friendly summary...\n")

    return {
        "search_results": "\n\n---\n\n".join(chunks),
        "sources": list(dict.fromkeys(sources)),  # de-duplicate, keep order
    }


# ---------- Node 5: summarize the search results ----------
def summarize(state: HealthBotState) -> dict:
    user_msg = (
        f"Health topic the patient asked about: {state['topic']}\n\n"
        f"SEARCH RESULTS (your only permitted source of information):\n"
        f"{state['search_results']}"
    )
    response = llm.invoke(
        [SystemMessage(content=SUMMARY_SYSTEM_PROMPT), HumanMessage(content=user_msg)]
    )
    summary = response.content.strip()

    return {
        "summary": summary,
        # Keep the summary in the conversation so later nodes/models can see it.
        "messages": [AIMessage(content=summary)],
    }


# ---------- Node 6: show the summary to the patient ----------
def display_summary(state: HealthBotState) -> dict:
    banner(f"WHAT YOU SHOULD KNOW ABOUT: {state['topic'].upper()}")
    print(wrap(state["summary"]))
    print("\nSources used:")
    for url in state["sources"]:
        print(f"  - {url}")
    return {}


# ---------- Node 7: wait until the patient is ready ----------
def readiness_check(state: HealthBotState) -> dict:
    print()
    input("Take your time reading. Press Enter when you're ready for a quick comprehension check... ")
    print("\nGreat - writing one question for you...\n")
    return {}


# ---------- Node 8: build the one-question quiz from the summary only ----------
def generate_quiz(state: HealthBotState) -> dict:
    user_msg = (
        "PATIENT-EDUCATION SUMMARY (your only permitted source of information):\n"
        f"{state['summary']}"
    )
    response = llm.invoke(
        [SystemMessage(content=QUIZ_SYSTEM_PROMPT), HumanMessage(content=user_msg)]
    )
    question = response.content.strip()

    return {
        "quiz_question": question,
        "messages": [AIMessage(content=f"Comprehension check question: {question}")],
    }


# ---------- Node 9: present the quiz question ----------
def present_quiz(state: HealthBotState) -> dict:
    banner("COMPREHENSION CHECK")
    print(wrap(state["quiz_question"]))
    return {}


# ---------- Node 10: collect the patient's answer ----------
def collect_answer(state: HealthBotState) -> dict:
    answer = input("\nYour answer: ").strip()
    if not answer:
        answer = "(no answer given)"
    print("\nGrading your answer...\n")

    return {
        "patient_answer": answer,
        "messages": [HumanMessage(content=f"My answer: {answer}")],
    }


# ---------- Node 11: grade the answer against the summary ----------
def grade_answer(state: HealthBotState) -> dict:
    user_msg = (
        "PATIENT-EDUCATION SUMMARY (your only source of truth):\n"
        f"{state['summary']}\n\n"
        f"QUIZ QUESTION:\n{state['quiz_question']}\n\n"
        f"PATIENT'S ANSWER:\n{state['patient_answer']}"
    )
    response = llm.invoke(
        [SystemMessage(content=GRADER_SYSTEM_PROMPT), HumanMessage(content=user_msg)]
    )
    feedback = response.content.strip()

    return {
        "grade_feedback": feedback,
        "messages": [AIMessage(content=feedback)],
    }


# ---------- Node 12: present the grade and feedback ----------
def present_grade(state: HealthBotState) -> dict:
    banner("YOUR RESULT")
    print(wrap(state["grade_feedback"]))
    return {}


# ---------- Node 13: new topic or exit? ----------
def ask_next_action(state: HealthBotState) -> dict:
    print()
    choice = ""
    while choice not in {"new", "exit"}:
        choice = input(
            "Would you like to learn about a NEW health topic, or EXIT? (new/exit): "
        ).strip().lower()
        if choice not in {"new", "exit"}:
            print("Please type 'new' or 'exit'.")
    return {"session_active": choice == "new"}


# ---------- Node 14: reset state for privacy before a new topic ----------
def reset_state(state: HealthBotState) -> dict:
    """Clear every piece of the previous topic, including the message history.

    RemoveMessage tells the `add_messages` reducer to delete those messages, so no previous
    health information (or the patient's answers) is carried into the next topic.
    """
    print("\nClearing the previous session's information...\n")

    cleared = blank_state()
    cleared["messages"] = [RemoveMessage(id=m.id) for m in state["messages"]]
    return cleared


# ---------- Node 15: goodbye ----------
def end_session(state: HealthBotState) -> dict:
    banner("SESSION ENDED")
    print("Thanks for using HealthBot. Remember: this is general health education,")
    print("not medical advice. Please talk to your own healthcare provider about your care.")
    return {"session_active": False}

## 6. Conditional edges

Two decisions are made in the graph:

* **after `search_agent`** — did the model emit a tool call? If yes, run the Tavily `ToolNode`;
  if the results are already in the conversation, move on to summarizing.
* **after `ask_next_action`** — reset the state and start a new topic, or end the session.

In [ ]:
def route_after_agent(state: HealthBotState) -> str:
    """Send the model's tool call to the Tavily ToolNode, otherwise move on."""
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tavily_tools"
    return "collect_results"


def route_after_next_action(state: HealthBotState) -> str:
    """Loop back for a new topic (after resetting state) or exit."""
    return "reset_state" if state["session_active"] else "end_session"

## 7. Build the graph

In [ ]:
builder = StateGraph(HealthBotState)

# --- register nodes ---
builder.add_node("collect_topic", collect_topic)
builder.add_node("search_agent", search_agent)
builder.add_node("tavily_tools", tavily_tool_node)
builder.add_node("collect_results", collect_results)
builder.add_node("summarize", summarize)
builder.add_node("display_summary", display_summary)
builder.add_node("readiness_check", readiness_check)
builder.add_node("generate_quiz", generate_quiz)
builder.add_node("present_quiz", present_quiz)
builder.add_node("collect_answer", collect_answer)
builder.add_node("grade_answer", grade_answer)
builder.add_node("present_grade", present_grade)
builder.add_node("ask_next_action", ask_next_action)
builder.add_node("reset_state", reset_state)
builder.add_node("end_session", end_session)

# --- wire the sequential workflow ---
builder.add_edge(START, "collect_topic")
builder.add_edge("collect_topic", "search_agent")

builder.add_conditional_edges(
    "search_agent",
    route_after_agent,
    {"tavily_tools": "tavily_tools", "collect_results": "collect_results"},
)
builder.add_edge("tavily_tools", "collect_results")

builder.add_edge("collect_results", "summarize")
builder.add_edge("summarize", "display_summary")
builder.add_edge("display_summary", "readiness_check")
builder.add_edge("readiness_check", "generate_quiz")
builder.add_edge("generate_quiz", "present_quiz")
builder.add_edge("present_quiz", "collect_answer")
builder.add_edge("collect_answer", "grade_answer")
builder.add_edge("grade_answer", "present_grade")
builder.add_edge("present_grade", "ask_next_action")

# --- loop or exit ---
builder.add_conditional_edges(
    "ask_next_action",
    route_after_next_action,
    {"reset_state": "reset_state", "end_session": "end_session"},
)
builder.add_edge("reset_state", "collect_topic")  # start over with a clean state
builder.add_edge("end_session", END)

healthbot = builder.compile()
print("HealthBot graph compiled.")

In [ ]:
# Visualize the workflow
from IPython.display import Image, display

try:
    display(Image(healthbot.get_graph().draw_mermaid_png()))
except Exception as exc:  # no network / no graphviz -> fall back to ASCII
    print("(Mermaid render unavailable:", exc, ")\n")
    print(healthbot.get_graph().draw_ascii())

## 8. Run HealthBot

Run the cell below and answer the prompts in the input boxes that appear.

* Enter any health topic, e.g. `type 2 diabetes`, `migraine`, `high blood pressure`, `asthma`.
* Press **Enter** when you have finished reading the summary.
* Type your answer to the quiz question.
* Type `new` to learn about another topic (state is reset first) or `exit` to finish.

`recursion_limit` is raised because each completed topic walks ~14 nodes and the graph loops.

In [ ]:
final_state = healthbot.invoke(blank_state(), config={"recursion_limit": 250})

## 9. Verifying state management and the privacy reset

`reset_state` runs on the **loop** edge — right before a new topic starts — so after the session
ends the state holds only the **most recent** topic. That is the proof: if you covered two or more
topics, nothing from the earlier ones survives, and the message count is one topic's worth (system
prompt, the patient's request, the tool call, the Tavily result, the summary, the quiz question, the
patient's answer, the grade) rather than the total across every topic.

In [ ]:
# What survived the session?
print("Messages remaining in state :", len(final_state["messages"]))
print("Topic in state              :", repr(final_state["topic"]))
print("Summary length              :", len(final_state["summary"]), "characters")
print("Quiz question               :", repr(final_state["quiz_question"][:70] + "..."))
print("Patient answer              :", repr(final_state["patient_answer"]))
print("Session active              :", final_state["session_active"])

print("
Message history still held in state:")
for m in final_state["messages"]:
    preview = str(m.content).replace("
", " ")[:58]
    tools = f" -> calls {m.tool_calls[0]['name']}" if getattr(m, "tool_calls", None) else ""
    print(f"  {type(m).__name__:<13} {preview}{tools}")

And here is `reset_state` applied directly to that final state — every field blanked, and one
`RemoveMessage` emitted per message so the `add_messages` reducer deletes the whole history. This is
exactly what runs before each new topic begins.

In [ ]:
cleared = reset_state(final_state)

for key, value in cleared.items():
    if key != "messages":
        print(f"{key:<16}: {value!r}")

print(f"{'messages':<16}: {len(cleared['messages'])} RemoveMessage objects "
      f"(one per message previously in state)")